In [ ]:
%load_ext autoreload
%autoreload 2

# Adjacency matrix

In [ ]:
#for problems with loguru and mpl3d
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from pygraphon.plots.plot_matrices import spy



# reload data to make sure no pollution from previous manipulations
G_gml = nx.read_gml("data/polblogs.gml", destringizer=int).to_undirected(reciprocal=False, as_view=False)

G_gml.remove_nodes_from(list(nx.isolates(G_gml)))

In [ ]:
A_raw = nx.to_numpy_array(G_gml)
A_raw -= np.diag(np.diag(A_raw))
A_2_block = nx.to_numpy_array(G_gml, nodelist=sorted(G_gml.nodes()))
# we remove the diagonal to avoid self-loops
A_2_block -= np.diag(np.diag(A_2_block))
degrees = dict(G_gml.degree())
nodes_0_sorted = sorted(G_gml.nodes(), key=lambda node: degrees[node], reverse=True)
gradient_A = nx.adjacency_matrix(G_gml, nodelist=nodes_0_sorted).toarray()
gradient_A -= np.diag(np.diag(gradient_A))


In [ ]:
# Compute the number of connection to the opposite party
def count_cross_connections(graph,node) -> int:
    neighbors = graph.neighbors(node[0])
    count = 0
    for neighbor in neighbors:
        if node[1] != graph.nodes[neighbor]["value"]:
            count += 1
    return count

inter_connections ={node[0]: count_cross_connections(G_gml,node) for node in G_gml.nodes(data="value")}

# Sort nodes by increasing degrees within each membership group
nodes_0 = [node[0] for node in G_gml.nodes(data="value") if node[1] == 0 ]
nodes_1 = [node[0]  for node in G_gml.nodes(data="value") if node[1] == 1]

nodes_0_sorted = sorted(nodes_0, key=lambda node: inter_connections[node], reverse=False)
nodes_1_sorted = sorted(nodes_1, key=lambda node: inter_connections[node], reverse=True)

# Create the reordered adjacency matrix
reordered_A_gml = nx.adjacency_matrix(G_gml, nodelist=nodes_0_sorted  + nodes_1_sorted).toarray()
reordered_A_gml -= np.diag(np.diag(reordered_A_gml))
reordered_A_gml[reordered_A_gml == 2] = 1

In [ ]:
marker_size = 0.2
fig, ax = plt.subplots(2, 2, figsize=(5, 5))
fig.subplots_adjust(right=0.9)
_ = spy(A_2_block, ax=ax[0,0], marker_size=marker_size, origin="lower")
_ = spy(gradient_A, ax=ax[0,1], marker_size=marker_size, origin="lower")
_ = spy(A_raw, ax=ax[1,0], marker_size=marker_size, origin="lower")
_ = spy(reordered_A_gml, ax=ax[1,1], marker_size=marker_size, origin="lower")

ax[0,0].set_title("Adjacency matrix")
#remove ticks and labels
ax[0,0].tick_params(top=False, bottom=False, left=False, right=False,
                labelleft=False, labelbottom=False)
ax[0,1].set_title("By degrees")
ax[0,1].tick_params(top=False, bottom=False, left=False, right=False,
                labelleft=False, labelbottom=False)
ax[1,0].set_title("By political party")
ax[1,0].tick_params(top=False, bottom=False, left=False, right=False,
                labelleft=False, labelbottom=False)
ax[1,1].set_title("By interconnectivty")
ax[1,1].tick_params(top=False, bottom=False, left=False, right=False,
                labelleft=False, labelbottom=False)
plt.show()

# Nethist run

In [ ]:
from pygraphon import HistogramEstimator, SmoothNetHist

est = SmoothNetHist()
est.fit(reordered_A_gml)
first_hist_olhede = est._first_approximate_graphon_from_adjacency(adjacencyMatrix= reordered_A_gml)
est_origin = HistogramEstimator()
est_origin.fit(reordered_A_gml)

print(est.get_ratio_par())
print("Number of unique parameters in SmoothNetHist:", len(np.unique(est.graphon.get_graphon())))
print("Number of unique parameters in Olhede:", len(np.unique(est_origin.get_graphon().graphon)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(est_origin.get_edge_connectivity(), cmap='jet')#, vmin=0, vmax=1)
im2 = ax[2].imshow(est.get_edge_connectivity(), cmap='jet')#, vmin=0, vmax=1)
_ = spy(reordered_A_gml, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("Olhede histogram")
ax[2].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(np.sqrt(est_origin.get_graphon().graphon), cmap='jet')#, vmin=0, vmax=1)
im2 = ax[2].imshow(np.sqrt(est.get_graphon().graphon), cmap='jet')#, vmin=0, vmax=1)
_ = spy(reordered_A_gml, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("Olhede histogram")
ax[2].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)

plt.show()

## Selecting at which point I should cut the graphon

In [ ]:
def print_array_with_indices(arr):
    for index, value in np.ndenumerate(arr):
        print(f"Index: {index}, Value: {value}")

print_array_with_indices(np.diag(est_origin.graphon.get_graphon()))

In [ ]:
turn_index = 8
end_index = est_origin.graphon.get_graphon().shape[0]
sort_index_dem = [turn_index - (i+1) for i in range(turn_index) ] + [end_index - (i+1) for i in range(end_index - turn_index)]
print(sort_index_dem)

## Permuting according to index

In [ ]:
graphon_polblogs = est.graphon.get_graphon()
# Get the diagonal elements
diagonal = np.diag(graphon_polblogs)

# Sort the diagonal elements in increasing order and get the sorting indices
sorting_indices = np.argsort(diagonal)
print(sorting_indices)

# Permute the rows and columns of the matrix based on the sorting indices
graphon_polblogs_permuted = graphon_polblogs[sort_index_dem][:, sort_index_dem]

print(graphon_polblogs_permuted.shape)

In [ ]:
fig = plt.figure(figsize=(6, 3.2))

ax = fig.add_subplot(111)
ax.set_title('colorMap')
plt.imshow(graphon_polblogs_permuted, cmap='jet')
ax.set_aspect('equal')

cax = fig.add_axes([0.12, 0.1, 0.78, 0.8])
cax.get_xaxis().set_visible(False)
cax.get_yaxis().set_visible(False)
cax.patch.set_alpha(0)
cax.set_frame_on(False)
plt.colorbar(orientation='vertical', ax = cax)
plt.show()

# 3d plots

In [ ]:
import numpy as np
sys.path.append("..")
from new_plots import plot_3d_and_reflection


# Plot the 3D plot and its reflection
plot_3d_and_reflection(graphon_polblogs_permuted*est.graphon.initial_rho)


In [ ]:
from mpl3d.camera import Camera
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from new_plots import Bar

Z = np.sqrt(graphon_polblogs_permuted).copy()*est.graphon.initial_rho

cmap = plt.get_cmap("jet")
norm = mpl.colors.Normalize(vmin=Z.min(),vmax=Z.max())
facecolors = cmap(norm(Z))


fig = plt.figure(figsize=(8,8), dpi=1200)
ax = fig.add_axes([0,0,1,1], xlim=[-1,+1], ylim=[-1,0], aspect=1)
ax.axis("off")

camera = Camera("perspective", 70, -135+90, scale=1)
bars = Bar(ax, camera.transform, Z, facecolors=facecolors)
camera.connect(ax, bars.update)

plt.show()

In [ ]:
fig2 = plt.figure(figsize=(8,8), dpi=1200)
ax2 = fig2.add_axes([0,0,1,1], xlim=[-1,+1], ylim=[-1,0], aspect=1)
ax2.axis("off")

camera = Camera("perspective", 70, -135+90, scale=1)
bars = Bar(ax2, camera.transform, np.zeros_like(Z), facecolors=facecolors)
camera.connect(ax, bars.update)

plt.show()